<a href="https://colab.research.google.com/github/mohitsbh/Amezon-/blob/main/distilbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd

API_KEY = "fe92a65b625840edbed4232a538250d8"

url = f"https://newsapi.org/v2/everything?q=technology&language=en&apiKey={API_KEY}"

response = requests.get(url)
data = response.json()

articles = []

for article in data["articles"]:
    articles.append({
        "title": article["title"],
        "description": article["description"],
        "content": article["content"],
        "source": article["source"]["name"],
        "label": 1   # 1 = REAL NEWS
    })

df = pd.DataFrame(articles)
df.to_csv("real_news.csv", index=False)

In [2]:
df=pd.read_csv("real_news.csv")
df.head()

,title,description,content,source,label
0,DLSS 5: Has Nvidia’s AI graphics technology go...,Nvidia has revealed a new “3D guided neural re...,Nvidia has revealed a new 3D guided neural ren...,The Verge,1
1,Signal’s Creator Is Helping Encrypt Meta AI,Moxie Marlinspike says the technology powering...,"Moxie Marlinspike, the privacy advocate who cr...",Wired,1
2,Video reviews to be introduced at Wimbledon,Video review technology will be introduced at ...,A review will also be allowed at the end of a ...,BBC News,1
3,"Really, you made this without AI? Prove it","""This looks like AI."" It's a phrase I dread se...",<ul><li></li><li></li><li></li></ul>\r\nHuman ...,The Verge,1
4,It’s not easy to get depression-detecting AI t...,"For the past seven years, the California-based...",<ul><li></li><li></li><li></li></ul>\r\nInstea...,The Verge,1


In [3]:
df.describe()

,label
count,83.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


In [4]:
df.describe()

,label
count,83.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


In [6]:
from google.colab import files
uploaded=files.upload()

Saving True.csv to True.csv


In [8]:
import pandas as pd

# 🔹 Load datasets
real = pd.read_csv("real_news.csv")
fake = pd.read_csv("Fake.csv")

# 🔹 Format FAKE dataset to match REAL
fake_df = pd.DataFrame()
fake_df["title"] = fake["title"]
fake_df["description"] = fake["text"].astype(str).str[:200]   # short summary
fake_df["content"] = fake["text"]
fake_df["label"] = 0   # FAKE

# 🔹 Keep only required columns in REAL
real_df = real[["title", "description", "content", "label"]]

# 🔹 Remove null values
real_df.dropna(inplace=True)
fake_df.dropna(inplace=True)

# 🔥 BALANCE DATASET (VERY IMPORTANT)
min_size = min(len(real_df), len(fake_df))

real_df = real_df.sample(n=min_size, random_state=42)
fake_df = fake_df.sample(n=min_size, random_state=42)

# 🔹 Combine datasets
final_df = pd.concat([real_df, fake_df])

# 🔹 Shuffle dataset
final_df = final_df.sample(frac=1, random_state=42)

# 🔹 Save final dataset
final_df.to_csv("final_dataset.csv", index=False)

print("✅ Final Dataset Size:", len(final_df))

✅ Final Dataset Size: 160


/tmp/ipykernel_2327/405518851.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  real_df.dropna(inplace=True)


In [9]:
import pandas as pd

# 🔹 Load datasets
api_real = pd.read_csv("real_news.csv")
kaggle_fake = pd.read_csv("Fake.csv")
kaggle_real = pd.read_csv("True.csv")

# -------------------------------
# 🔹 FORMAT API REAL
# -------------------------------
api_real_df = api_real[["title", "description", "content"]].copy()
api_real_df["label"] = 1

# -------------------------------
# 🔹 FORMAT KAGGLE REAL
# -------------------------------
kaggle_real_df = pd.DataFrame()
kaggle_real_df["title"] = kaggle_real["title"]
kaggle_real_df["description"] = kaggle_real["text"].astype(str).str[:200]
kaggle_real_df["content"] = kaggle_real["text"]
kaggle_real_df["label"] = 1

# -------------------------------
# 🔹 FORMAT KAGGLE FAKE
# -------------------------------
kaggle_fake_df = pd.DataFrame()
kaggle_fake_df["title"] = kaggle_fake["title"]
kaggle_fake_df["description"] = kaggle_fake["text"].astype(str).str[:200]
kaggle_fake_df["content"] = kaggle_fake["text"]
kaggle_fake_df["label"] = 0

# -------------------------------
# 🔹 COMBINE ALL REAL DATA
# -------------------------------
real_df = pd.concat([api_real_df, kaggle_real_df])

# 🔹 Remove nulls
real_df.dropna(inplace=True)
kaggle_fake_df.dropna(inplace=True)

# -------------------------------
# 🔥 BALANCE DATASET
# -------------------------------
min_size = min(len(real_df), len(kaggle_fake_df))

real_df = real_df.sample(n=min_size, random_state=42)
fake_df = kaggle_fake_df.sample(n=min_size, random_state=42)

# -------------------------------
# 🔹 FINAL COMBINE
# -------------------------------
final_df = pd.concat([real_df, fake_df])

# 🔹 Shuffle
final_df = final_df.sample(frac=1, random_state=42)

# 🔹 Create text column (important for ML)
final_df["text"] = final_df["title"] + " " + final_df["content"]

# 🔹 Save
final_df.to_csv("final_dataset.csv", index=False)

print("✅ Final dataset created:", len(final_df))

✅ Final dataset created: 42994


In [10]:
print(final_df["label"].value_counts())

label
0    21497
1    21497
Name: count, dtype: int64


In [11]:
import pandas as pd
import re

# Load final dataset
df = pd.read_csv("final_dataset.csv")

# 🔹 Clean function
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+", "", text)      # remove URLs
    text = re.sub(r"[^a-zA-Z ]", "", text)   # remove special chars
    text = re.sub(r"\s+", " ", text)         # remove extra spaces
    return text

# Apply cleaning
df["text"] = df["text"].apply(clean_text)

print("✅ Text cleaned")

✅ Text cleaned


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

X = df["text"]
y = df["label"]

# 🔹 Convert text → vectors
vectorizer = TfidfVectorizer(max_features=5000)

X_vec = vectorizer.fit_transform(X)

print("✅ Text converted to vectors")

✅ Text converted to vectors


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.2, random_state=42
)

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("✅ Model trained")

✅ Model trained


In [14]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9873241074543552
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4277
           1       0.99      0.99      0.99      4322

    accuracy                           0.99      8599
   macro avg       0.99      0.99      0.99      8599
weighted avg       0.99      0.99      0.99      8599



In [15]:
import pickle

pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

print("✅ Model saved")

✅ Model saved


In [16]:
import pandas as pd

df = pd.read_csv("final_dataset.csv")

# Use only required columns
df = df[["text", "label"]]

print(df.head())

                                                text  label
0  WHY THIS REPUBLICAN GOVERNOR Is Being Called “...      0
1  Trump says U.S. not 'putting up with' North Ko...      1
2   Trump Tries To Blame Obama For North Korea Me...      0
3  WIKILEAKS REMINDS THE WORLD: “Obama has a hist...      0
4  U.S. majority backs military action vs. North ...      1


In [17]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)

In [18]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=256
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [19]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=256
)

In [20]:
import torch

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
val_dataset = NewsDataset(val_encodings, val_labels)

In [21]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,                 # ⚡ fast
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    logging_dir="./logs",
    save_strategy="epoch",
    fp16=True   # ⚡ faster on GPU
)

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [23]:
!pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 66.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [1]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs"
)

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [2]:
!pip uninstall -y transformers
!pip install transformers==4.36.2 accelerate -q

Found existing installation: transformers 5.5.4
Uninstalling transformers-5.5.4:
  Successfully uninstalled transformers-5.5.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.


In [1]:
import transformers
print(transformers.__version__)

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

4.36.2


In [2]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",      # ✅ NEW PARAM NAME
    save_strategy="epoch",
    logging_dir="./logs"
)

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'eval_strategy'

In [3]:
import pandas as pd

df = pd.read_csv("/content/final_dataset.csv")

# Use only needed columns
df = df[["text", "label"]]

# ⚡ Reduce size for faster training (optional but recommended)
df = df.sample(10000, random_state=42)

df.head()

,text,label
32193,Hawaii to resume Cold War-era nuclear siren te...,1
21379,House panel presses White House for fuller res...,1
42275,"Trump Just Went Back On Another Election Vow,...",0
29885,Ex-Illinois congressman convicted of failing t...,1
8194,Legal Community SCHOOLS Trump’s Lawyer For Cl...,0


In [4]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)

In [5]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=128
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
import torch

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
val_dataset = NewsDataset(val_encodings, val_labels)

In [7]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'pre_classifier.bias', 'pre_classifier.weight', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",     # ✅ correct for your version
    save_strategy="epoch",
    logging_dir="./logs",
    fp16=True                  # ⚡ GPU speed
)

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'eval_strategy'

In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs"
)

In [10]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
cannot import name 'EncoderDecoderCache' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

In [11]:
!pip uninstall -y transformers peft accelerate

Found existing installation: transformers 4.36.2
Uninstalling transformers-4.36.2:
  Successfully uninstalled transformers-4.36.2
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0


In [12]:
!pip install transformers==4.36.2 peft==0.7.1 accelerate==0.25.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 11.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.


In [1]:
import transformers, peft

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Transformers: 4.36.2
PEFT: 0.7.1


In [2]:
from transformers import Trainer

In [3]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs"
)

In [5]:
import pandas as pd

df = pd.read_csv("/content/final_dataset.csv")

# Use required columns
df = df[["text", "label"]]

# Optional: reduce size for faster training
df = df.sample(10000, random_state=42)

df.head()

,text,label
32193,Hawaii to resume Cold War-era nuclear siren te...,1
21379,House panel presses White House for fuller res...,1
42275,"Trump Just Went Back On Another Election Vow,...",0
29885,Ex-Illinois congressman convicted of failing t...,1
8194,Legal Community SCHOOLS Trump’s Lawyer For Cl...,0


In [6]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)

In [7]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=128
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
import torch

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
val_dataset = NewsDataset(val_encodings, val_labels)

In [9]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs"
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [16]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

Step,Training Loss
500,0.049800
1000,0.007700
1500,0.005100
2000,0.002500


TrainOutput(global_step=2000, training_loss=0.016287220895290375, metrics={'train_runtime': 265.6331, 'train_samples_per_second': 60.233, 'train_steps_per_second': 7.529, 'total_flos': 529869594624000.0, 'train_loss': 0.016287220895290375, 'epoch': 2.0})

In [17]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=1)

print("Accuracy:", accuracy_score(val_labels, preds))
print(classification_report(val_labels, preds))

Accuracy: 0.999
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1015
           1       1.00      1.00      1.00       985

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [18]:
model.save_pretrained("/content/distilbert_model")
tokenizer.save_pretrained("/content/distilbert_model")

('/content/distilbert_model/tokenizer_config.json',
 '/content/distilbert_model/special_tokens_map.json',
 '/content/distilbert_model/vocab.txt',
 '/content/distilbert_model/added_tokens.json')

In [19]:
import shutil

shutil.make_archive("/content/distilbert_model", 'zip', "/content/distilbert_model")

'/content/distilbert_model.zip'

In [20]:
from google.colab import files

files.download("/content/distilbert_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>